# Apply the plate-position correction to single-cell profiles

Wells are shifted by where they sit on the plate (a "tilt"), independent of what is in the well.
The tilt was estimated from all plates in [`3a.position_correction/fit_position_correction.ipynb`](3a.position_correction/fit_position_correction.ipynb) and saved to `3a.position_correction/position_correction_fit.npz`.

This step applies that saved fit to each plate's normalized single-cell profiles, then performs feature selection on the corrected profiles.

For every cell, the correction subtracts `amplitude x tilt(row, column)` from the features, where the amplitude is specific to the platemap.
All cells in a well are shifted by the same amount, so differences between cells in a well are unchanged.
Controls are corrected like all other wells.
Features that are not part of the fit are left unchanged.

**Inputs** (per plate, from step 2): `<plate>_sc_normalized.parquet`

**Outputs** (per plate, in `single_cell_profiles/`):

- `<plate>_sc_position_corrected.parquet`: normalized profiles after correction
- `<plate>_sc_position_corrected_feature_selected.parquet`: corrected profiles after feature selection

The uncorrected `<plate>_sc_feature_selected.parquet` from step 2 is not changed, so the two versions can be compared.

Run `3a.position_correction/fit_position_correction.ipynb` first when the normalized profiles change.

## Import libraries

In [1]:
import os
import pathlib
import pprint
import sys

import pandas as pd
from pycytominer import feature_select

sys.path.append("../utils")
import position_correction_utils as pcu

## Set paths and variables

In [2]:
# get the batch to process from environment variable
batch_to_process = os.environ.get("BATCH", "batch_1")

# base directory where batches are located
base_dir = pathlib.Path("./data/").resolve(strict=True)

# Decide what to process
if batch_to_process:
    print(f"Processing {batch_to_process}")
    batch_dirs = [base_dir / batch_to_process]
else:
    print("No specific batch set, processing all available batches")
    batch_dirs = [p for p in base_dir.glob("batch_*") if p.is_dir()]

# position correction fit written by 3a.position_correction/fit_position_correction.ipynb
fit_path = pathlib.Path("./3a.position_correction/position_correction_fit.npz").resolve(
    strict=True
)

# operations to perform for feature selection (same as step 2)
# pycytominer applies them in this order, so we first drop features with missing
# values and blocklisted features, and then apply the variance and correlation filters
feature_select_ops = [
    "drop_na_columns",
    "blocklist",
    "variance_threshold",
    "correlation_threshold",
]

Processing batch_1


## Load the position correction fit

The fit holds the tilt map for the features it was estimated on and one amplitude per platemap.

In [3]:
fit = pcu.load_fit(fit_path)

print("Platemaps in fit:", sorted(fit["amplitudes"]))
print("Features in fit:", len(fit["features"]))

Platemaps in fit: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Features in fit: 2446


## Set dictionary with plates to process

The platemap number of each plate comes from its folder name (`platemap_N`).
Processing stops if a platemap has no amplitude in the fit.

In [4]:
plate_info_dictionary = {}

for batch_dir in batch_dirs:
    for layout_dir in sorted(p for p in batch_dir.iterdir() if p.is_dir()):
        platemap = int(layout_dir.name.split("_")[1])
        if platemap not in fit["amplitudes"]:
            raise ValueError(
                f"No amplitude for {layout_dir.name} in the position correction fit. "
                "Rerun 3a.position_correction/fit_position_correction.ipynb."
            )

        profile_dir = layout_dir / "single_cell_profiles"
        for normalized_path in sorted(profile_dir.glob("*_sc_normalized.parquet")):
            plate = normalized_path.name.removesuffix("_sc_normalized.parquet")
            plate_info_dictionary[plate] = {
                "normalized_path": normalized_path,
                "platemap": platemap,
                "output_dir": profile_dir,
            }

# View dictionary
print("Number of plates to process:", len(plate_info_dictionary))
pprint.pprint(plate_info_dictionary, indent=4)

Number of plates to process: 16
{   'CARD-CelIns-CX7_251023210001': {   'normalized_path': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles/CARD-CelIns-CX7_251023210001_sc_normalized.parquet'),
                                        'output_dir': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles'),
                                        'platemap': 1},
    'CARD-CelIns-CX7_251124150001': {   'normalized_path': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles/CARD-CelIns-CX7_251124150001_sc_normalized.parquet'),
                                        'output_dir': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles'),
                                        'platemap': 1},
    'CARD-CelIns-CX7_2511251

## Correct profiles and perform feature selection

Feature selection runs on the corrected profiles with the same settings as step 2.

In [5]:
for plate, info in plate_info_dictionary.items():
    output_dir = info["output_dir"]
    output_corrected_file = output_dir / f"{plate}_sc_position_corrected.parquet"
    output_feature_select_file = (
        output_dir / f"{plate}_sc_position_corrected_feature_selected.parquet"
    )

    print("Applying position correction to", plate, "(platemap", info["platemap"], ")")
    normalized_df = pd.read_parquet(info["normalized_path"])

    # Step 1: subtract the platemap-scaled tilt from every cell
    corrected_df = pcu.correct_cells(normalized_df, fit, info["platemap"])
    n_corrected = len([f for f in fit["features"] if f in normalized_df.columns])
    print(f"Corrected {n_corrected} features in {len(corrected_df)} cells")
    corrected_df.to_parquet(output_corrected_file, index=False)

    # Step 2: feature selection on the corrected profiles
    print("Performing feature selection for", plate, "...")
    feature_select(
        profiles=corrected_df,
        operation=feature_select_ops,
        na_cutoff=0,
        output_file=str(output_feature_select_file),
        output_type="parquet",
        blocklist_file="./blocklist_features.txt",
    )

    print(f"Position correction and feature selection complete for {plate}")

Applying position correction to CARD-CelIns-CX7_251023210001 (platemap 1 )


Corrected 2446 features in 7839 cells


Performing feature selection for CARD-CelIns-CX7_251023210001 ...


Position correction and feature selection complete for CARD-CelIns-CX7_251023210001
Applying position correction to CARD-CelIns-CX7_251124150001 (platemap 1 )


Corrected 2446 features in 8456 cells


...

Performing feature selection for CARD-CelIns-CX7_251212180001 ...


Position correction and feature selection complete for CARD-CelIns-CX7_251212180001
Applying position correction to CARD-CelIns-CX7_251213150001 (platemap 4 )


Corrected 2446 features in 9027 cells


Performing feature selection for CARD-CelIns-CX7_251213150001 ...


Position correction and feature selection complete for CARD-CelIns-CX7_251213150001


## Check outputs

In [6]:
# Check the last plate: corrected profiles keep the cells and metadata of the normalized profiles
normalized_df = pd.read_parquet(info["normalized_path"])
corrected_df = pd.read_parquet(output_corrected_file)
selected_df = pd.read_parquet(output_feature_select_file)

metadata_cols = [c for c in normalized_df.columns if c.startswith("Metadata_")]
print("Normalized:", normalized_df.shape)
print("Corrected:", corrected_df.shape)
print("Corrected and feature selected:", selected_df.shape)
print(
    "Metadata unchanged:",
    normalized_df[metadata_cols].equals(corrected_df[metadata_cols]),
)

feature_cols = [f for f in fit["features"] if f in normalized_df.columns]
shift = (normalized_df[feature_cols] - corrected_df[feature_cols]).abs()
print(f"Mean absolute shift per feature: {shift.to_numpy().mean():.3f}")
selected_df.head(2)

Normalized: (9027, 2477)
Corrected: (9027, 2477)
Corrected and feature selected: (9027, 1142)
Metadata unchanged: True


Mean absolute shift per feature: nan


,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Pathway,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,...,Nuclei_Texture_InverseDifferenceMoment_ER_3_00_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_02_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_02_256,Nuclei_Texture_SumEntropy_Mito_3_00_256,Nuclei_Texture_SumVariance_Actin_3_03_256,Nuclei_Texture_SumVariance_DNA_3_01_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Mito_3_03_256,Nuclei_Texture_SumVariance_PM_3_01_256
0,B,2,7,healthy,None,DMSO,None,616.334182,191.524507,612.833657,...,1.008522,0.584461,0.56229,1.465914,0.172893,-0.343721,-0.172644,-0.330715,-0.236212,-0.281784
1,B,2,7,healthy,None,DMSO,None,150.808147,61.379106,217.762201,...,0.394595,-0.028498,0.37508,1.158001,-0.619304,-0.556596,0.313870,-0.285745,-0.263332,-0.295998
